<a href="https://colab.research.google.com/github/Chanthul4054/E-Policing-An-Integrated-Spatio-Temporal-Crime-Forecasting-and-Decision-Support-System/blob/Spatio-Temporal-Crime-Prediction-System/Component_1_DPP%26DC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Component based data preprocessing

"To generate a weekly probability score (0-1) for every GN Division, specific to a user-selected crime type, by analyzing historical trends, land use, and spatial lags."

Historical Analysis (Past Week): A diagnostic tool that shows exactly what happened, allowing users to compare the "Predicted" risk against "Actual" results.

##Preprocessing & Data Cleaning steps performed


Remove Irrelevant Columns

Handling Typos/Inconsistencies

Coordinate Validation

Datetime Conversion

Boolean Mapping

Categorical Encoding

Temporal Decomposition



#EDA

Temporal Validation (Validating your Datetime Conversions)

*   Plot a Crime Count Trend Line over time. This validates your "Datetime Conversion" and checks for seasonal spikes.
*   Use a Heatmap for Time of Day vs. Day of Week. This visually justifies why your model needs the "Hour" and "Day_of_Week" features you created during preprocessing.

Spatial Validation (Validating Coordinate & GN Division Cleaning)

*   Create a Geospatial Scatter Plot using latitude and longitude. If any points appear outside your expected region (e.g., in the ocean), it means your "Coordinate Validation" step needs further refinement.
*   Use a Bar Chart for Crime Density by GN Division. This highlights the "Natural Hotspots" in your historical data, which serves as a baseline for your "Crime Hotspot Prediction" task.

Categorical & Demographic Validation (Validating Encoding & Mapping)

*   Create Count Plots for Crime Types. Given that you have 1,608 rows, you need to see if there is a "Class Imbalance". For example, if "Burglary" is 80% of your data, your "Crime Type Classification" model will be biased.
*   Plot Victim Age Distribution (Histogram). This validates your "Handling Typos" step by ensuring no ages are impossible (e.g., 200 years old).







##Component Based Preprocessing

In [121]:
import pandas as pd
import numpy as np

In [122]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [123]:
crime_df = pd.read_csv('/content/drive/MyDrive/DSGP/CrimeData_v3.csv')

In [124]:
crime_df

,crime,location,date,sex,victim_ethnicity,victim_age,time,weather,latitude,longitude,holiday_name,is_holiday,land_use_type,gn_division,gn_pcode,gn_population,gn_distance_m,victim_ethnicity,status_report
0,burglary,mulgampola,2019-12-31,f,muslim,54,08:17:00,Cloudy,7.280544,80.616500,Non-Holiday,0,General Urban,Welata,LK2130170,21826,311.8,muslim,Valid
1,burglary,car park,2020-01-04,m,sinhala,42,02:00:00,Rainy,7.283445,80.619385,Non-Holiday,0,Commercial,Katukele West,LK2130105,8913,352.4,sinhala,Valid
2,theft,bus stand,2020-01-08,f,sinhala,20,21:01:00,Rainy,7.256425,80.590461,Eid al-Adha,1,Commercial,Penideniya,LK2139135,16411,518.6,sinhala,Valid
3,vehicle theft,aniwatte,2020-01-10,m,sinhala,29,12:10:00,Cloudy,7.290058,80.622438,Adhi Vap Full Moon Poya Day,1,General Urban,Aniwatta East,LK2130100,1107,381.5,sinhala,Valid
4,robbery,dutugamunu mawatha,2020-01-11,m,sinhala,59,02:39:00,Rainy,7.312344,80.645687,Non-Holiday,0,General Urban,Aruppala East,LK2130050,1293,322.6,sinhala,Valid
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1603,drugs,walespark,2025-07-30,m,sinhala,65,09:19:00,Rainy,7.291772,80.636707,Non-Holiday,0,Commercial,Mahanuwara,LK2130120,11613,518.0,sinhala,Valid
1604,drugs,peradeniya road,2025-08-16,m,sinhala,59,04:38:00,Rainy,7.280930,80.619477,Non-Holiday,0,General Urban,Suduhumpala West,LK2130165,18808,353.7,sinhala,Valid
1605,robbery,provincial education department,2025-09-13,f,sinhala,50,07:25:00,Rainy,7.290341,80.633563,Non-Holiday,0,Commercial,Bogambara,LK2130145,2198,315.0,sinhala,Valid
1606,robbery,post office,2025-09-17,f,sinhala,61,20:12:00,Rainy,7.292186,80.633475,Non-Holiday,0,Commercial,Ihala Katukele,LK2130115,6925,335.7,sinhala,Valid


In [125]:
crime_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1608 entries, 0 to 1607
Data columns (total 19 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   crime              1608 non-null   object 
 1   location           1608 non-null   object 
 2   date               1608 non-null   object 
 3   sex                1608 non-null   object 
 4   victim_ethnicity   1608 non-null   object 
 5   victim_age         1608 non-null   int64  
 6   time               1608 non-null   object 
 7   weather            1608 non-null   object 
 8   latitude           1608 non-null   float64
 9   longitude          1608 non-null   float64
 10  holiday_name       1608 non-null   object 
 11  is_holiday         1608 non-null   int64  
 12  land_use_type      1608 non-null   object 
 13  gn_division        1608 non-null   object 
 14  gn_pcode           1608 non-null   object 
 15  gn_population      1608 non-null   int64  
 16  gn_distance_m      1608 

In [126]:
crime_df.describe()

,victim_age,latitude,longitude,is_holiday,gn_population,gn_distance_m
count,1608.000000,1608.000000,1608.000000,1608.000000,1608.000000,1608.000000
mean,42.602612,7.285506,80.633042,0.070896,8630.593284,398.776306
std,13.871646,0.025415,0.021201,0.256730,6767.678765,640.411182
min,15.000000,7.094806,80.528761,0.000000,926.000000,6.700000
25%,32.000000,7.279031,80.624353,0.000000,2198.000000,220.675000
50%,42.000000,7.289164,80.634393,0.000000,6925.000000,316.150000
75%,52.000000,7.294233,80.639486,0.000000,13050.000000,445.600000
max,78.000000,7.499662,80.764916,1.000000,22268.000000,8483.300000


###  Temporal Decomposition

In [127]:
crime_df['date'] = pd.to_datetime(crime_df['date'])

# Extract the month
crime_df['month'] = crime_df['date'].dt.month

# Extract the day of the week
crime_df['day_of_week'] = crime_df['date'].dt.dayofweek

# Create 'is_weekend
crime_df['is_weekend'] = crime_df['day_of_week'].apply(lambda x: 1 if x >= 5 else 0)

# Extract the year
crime_df['year'] = crime_df['date'].dt.year

### Time-Binning

Give every crime in your original list a label (Morning, Night, etc.).

In [128]:
#Create time bins-> devide the 24-hour day in to blocks

#Extract hour and create bins
crime_df['hour'] = pd.to_datetime(crime_df['time']).dt.hour

# Define bins 0-6 (Night), 6-12 (Morning), 12-18 (Afternoon), 18-24 (Evening)
bins = [0, 6, 12, 18, 24]
labels = ['Night', 'Morning', 'Afternoon', 'Evening']
crime_df['time_bin'] = pd.cut(crime_df['hour'], bins=bins, labels=labels, right=False)

crime_df['time_bin'] = pd.Categorical(crime_df['time_bin'], categories=labels, ordered=True)

/tmp/ipython-input-3875019255.py:4: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  crime_df['hour'] = pd.to_datetime(crime_df['time']).dt.hour


### Spatial Feature Aggregation

This code converts your list of individual crime incidents into a neat summary table that shows the total number of crimes for each neighborhood on each day.

In [129]:
#The primary key -> gn_pcode
# Convert date to a weekly period
crime_df['week_start'] = pd.to_datetime(crime_df['date']).dt.to_period('W').dt.start_time
# Group by Date and GN Division
weekly_aggregated = crime_df.groupby([
    'week_start', 'gn_pcode', 'gn_division', 'gn_population',
    'land_use_type'
], observed=True).size().reset_index(name='total_crimes')
# Sort by date and division for a clear timeline
weekly_aggregated = weekly_aggregated.sort_values(by=['week_start', 'gn_division'])



In [130]:
# Display the result
print("First 10 rows of Aggregated Data:")
print(weekly_aggregated.head(10))

First 10 rows of Aggregated Data:
   week_start   gn_pcode       gn_division  gn_population  land_use_type  \
0  2019-12-30  LK2130105     Katukele West           8913     Commercial   
1  2019-12-30  LK2130170            Welata          21826  General Urban   
3  2020-01-06  LK2130100     Aniwatta East           1107  General Urban   
2  2020-01-06  LK2130050     Aruppala East           1293  General Urban   
5  2020-01-06  LK2139135        Penideniya          16411     Commercial   
4  2020-01-06  LK2130165  Suduhumpala West          18808     Commercial   
7  2020-01-13  LK2130085          Asgiriya           1314    Residential   
9  2020-01-13  LK2130135       Boowelikada           2310     Commercial   
10 2020-01-13  LK2130205            Bowala           2430     Commercial   
6  2020-01-13  LK2130065         Mahaiyawa          11543     Commercial   

    total_crimes  
0              1  
1              1  
3              1  
2              1  
5              1  
4              

### Grid Sampling

In [131]:
# 1. Prepare unique values
all_weeks = weekly_aggregated['week_start'].unique()
all_gn_codes = weekly_aggregated['gn_pcode'].unique() # This is a numpy array
all_gn = weekly_aggregated[['gn_pcode', 'gn_division', 'gn_population', 'land_use_type']].drop_duplicates()

# 2. CREATE THE MASTER GRID
index = pd.MultiIndex.from_product(
    [all_weeks, all_gn_codes],
    names=['week_start', 'gn_pcode']
)
master_grid = pd.DataFrame(index=index).reset_index()

crime_pivot = crime_df.pivot_table(
    index=['week_start', 'gn_pcode'],
    columns='crime',
    aggfunc='size',
    fill_value=0
).reset_index()

# Pivot Time Bins into columns as well (context for the model)
time_pivot = crime_df.pivot_table(
    index=['week_start', 'gn_pcode'],
    columns='time_bin',
    aggfunc='size',
    fill_value=0
).reset_index()

final_grid = pd.merge(master_grid, all_gn, on='gn_pcode', how='left')
final_grid = pd.merge(final_grid, weekly_aggregated[['week_start', 'gn_pcode', 'total_crimes']], on=['week_start', 'gn_pcode'], how='left')
final_grid = pd.merge(final_grid, crime_pivot, on=['week_start', 'gn_pcode'], how='left')
final_grid = pd.merge(final_grid, time_pivot, on=['week_start', 'gn_pcode'], how='left')

final_grid = final_grid.fillna(0)

# Create Target: Did any crime happen NEXT week?
final_grid = final_grid.sort_values(['gn_pcode', 'week_start'])
final_grid['target_next_week'] = final_grid.groupby('gn_pcode')['total_crimes'].shift(-1)
final_grid['target_next_week'] = (final_grid['target_next_week'] > 0).astype(int)


/tmp/ipython-input-386170819.py:21: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  time_pivot = crime_df.pivot_table(


### Trend and Seasonality Extraction




In [132]:
#1. Extract Basic Time Components

# Extract basic features from the 'date' column
final_grid['month'] = final_grid['week_start'].dt.month

#2. Map the Season - > weather patterns
def get_season(month):
    if month in [12, 1, 2]: return 'NE_Monsoon'
    if month in [3, 4]: return 'First_Inter'
    if month in [5, 6, 7, 8, 9]: return 'SW_Monsoon'
    return 'Second_Inter'

final_grid['season'] = final_grid['month'].apply(get_season)
final_grid = pd.get_dummies(final_grid, columns=['season'], prefix='season')

# 3. Weekly Holiday Flag
# We check if the specific week contained a holiday in your original data
crime_df['week_start'] = pd.to_datetime(crime_df['date']).dt.to_period('W').dt.start_time
weeks_with_holidays = crime_df[crime_df['is_holiday'] == 1]['week_start'].unique()

final_grid['has_holiday_in_week'] = final_grid['week_start'].isin(weeks_with_holidays).astype(int)

# 4. Long-term Trend Indicator (Weeks from Start)
start_week = final_grid['week_start'].min()
final_grid['weeks_from_start'] = ((final_grid['week_start'] - start_week).dt.days // 7).astype(int)

# Calculate a 4-week rolling average of total crimes
final_grid['crime_trend_4w'] = final_grid.groupby('gn_pcode')['total_crimes'].transform(
    lambda x: x.rolling(window=4, min_periods=1).mean()
)

In [133]:
final_grid.info()

<class 'pandas.core.frame.DataFrame'>
Index: 29253 entries, 80 to 29240
Data columns (total 25 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   week_start           29253 non-null  datetime64[ns]
 1   gn_pcode             29253 non-null  object        
 2   gn_division          29253 non-null  object        
 3   gn_population        29253 non-null  int64         
 4   land_use_type        29253 non-null  object        
 5   total_crimes         29253 non-null  float64       
 6   burglary             29253 non-null  float64       
 7   drugs                29253 non-null  float64       
 8   robbery              29253 non-null  float64       
 9   stabbing             29253 non-null  float64       
 10  theft                29253 non-null  float64       
 11  vehicle theft        29253 non-null  float64       
 12  Night                29253 non-null  int64         
 13  Morning              29253 non-null

In [134]:
final_grid.describe()

,week_start,gn_population,total_crimes,burglary,drugs,robbery,stabbing,theft,vehicle theft,Night,Morning,Afternoon,Evening,target_next_week,month,has_holiday_in_week,weeks_from_start,crime_trend_4w
count,29253,29253.000000,29253.000000,29253.000000,29253.000000,29253.000000,29253.000000,29253.000000,29253.000000,29253.000000,29253.000000,29253.000000,29253.000000,29253.000000,29253.000000,29253.000000,29253.000000,29253.000000
mean,2022-08-30 20:33:56.468054528,10688.255119,0.129183,0.043346,0.027689,0.022254,0.021297,0.030014,0.010597,0.018562,0.040064,0.053430,0.043141,0.112843,6.458859,0.226370,139.265272,0.129266
min,2019-12-30 00:00:00,926.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000
25%,2021-05-03 00:00:00,2430.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,3.000000,0.000000,70.000000,0.000000
50%,2022-08-29 00:00:00,11008.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,6.000000,0.000000,139.000000,0.000000
75%,2023-12-25 00:00:00,18568.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,9.000000,0.000000,208.000000,0.000000
max,2025-09-22 00:00:00,22268.000000,5.000000,3.000000,2.000000,2.000000,2.000000,2.000000,2.000000,2.000000,3.000000,3.000000,3.000000,1.000000,12.000000,1.000000,299.000000,3.000000
std,NaN,7432.630795,0.387696,0.239724,0.175169,0.155850,0.155105,0.183189,0.108556,0.137236,0.220097,0.259872,0.230158,0.316406,3.435711,0.418488,80.816602,0.312729


Shows if crime frequency actually changes over time

### Spatio-Temporal Engineering

In [135]:
# Sort to ensure time is linear for each GN and Crime Type
final_grid = final_grid.sort_values(by=['gn_pcode', 'week_start'])

# 1. Temporal Lags (What happened 1 and 2 weeks ago?)
final_grid['lag_count_1w'] = final_grid.groupby('gn_pcode')['total_crimes'].shift(1).fillna(0)
final_grid['lag_count_2w'] = final_grid.groupby('gn_pcode')['total_crimes'].shift(2).fillna(0)
final_grid['crime_trend_4w'] = final_grid.groupby('gn_pcode')['crime_trend_4w'].shift(1).fillna(0)
#  Create a 'District Proxy' from gn_pcode
final_grid['district_id'] = final_grid['gn_pcode'].astype(str).str[:4]

# Calculate the average crime for that entire district during that specific week/time
dist_avg = final_grid.groupby(['week_start','district_id'])['total_crimes'].transform('mean')

# Shift the district average by 1 week so the model sees what happened nearby LAST week
final_grid['spatial_lag_1w'] = final_grid.groupby('gn_pcode')[dist_avg.name].shift(1).fillna(0)

# 2. Target Label (What will happen NEXT week? - This is what we predict)
final_grid['target_next_week'] = (
    final_grid.groupby(['gn_pcode'])['total_crimes'].shift(-1) > 0
).astype(int)

# Fill the resulting NaNs (from shifts) with 0 and drop temporary columns
final_grid[['lag_count_1w', 'lag_count_2w', 'spatial_lag_1w']] = final_grid[['lag_count_1w', 'lag_count_2w', 'spatial_lag_1w']].fillna(0)
final_grid = final_grid.drop(columns=['district_id'])


### Encoding Categorical Variables

In [136]:
cols_to_encode = ['land_use_type', 'season']

existing_categorical = [c for c in ['land_use_type'] if c in final_grid.columns]

# 1. One-Hot Encode
final_grid = pd.get_dummies(final_grid, columns=existing_categorical)

### NaN Scrubbing

In [137]:
lag_cols = ['lag_count_1w', 'lag_count_2w', 'spatial_lag_1w']

print(f"Rows before scrubbing: {len(final_grid)}")
# We drop NaNs here because the model cannot process 'None' values in these features
final_grid = final_grid.dropna(subset=lag_cols)
print(f"Rows after scrubbing: {len(final_grid)}")

Rows before scrubbing: 29253
Rows after scrubbing: 29253


### Seperate the Inference Row

In [138]:
# The very last week in your data is for PREDICTION, the rest is for TRAINING
latest_week = final_grid['week_start'].max()

# Data to show on the "Next Week" Heatmap
inference_data = final_grid[final_grid['week_start'] == latest_week].copy()

# Data to teach the model
train_df = final_grid[final_grid['week_start'] < latest_week].copy()

### Balanced Sampling

In [139]:
# 1. Separate classes ONLY from the training portion
pos = train_df[train_df['target_next_week'] == 1]
neg = train_df[train_df['target_next_week'] == 0]

# 2. Create the balanced training set
balanced_train = pd.concat([
    pos,
    neg.sample(len(pos) * 3, random_state=42)
])

# 3. Shuffle the result so the model doesn't see all 1s followed by all 0s
balanced_train = balanced_train.sample(frac=1, random_state=42).reset_index(drop=True)

In [140]:
print("Balanced Training Distribution:")
print(balanced_train['target_next_week'].value_counts(normalize=True))

Balanced Training Distribution:
target_next_week
0    0.75
1    0.25
Name: proportion, dtype: float64


### Temporal Train-Test Split

In [141]:
# Define the features to EXCLUDE
exclude_cols = [
    'gn_division',
    'week_start',
    'gn_pcode',
    'target_next_week',
    'total_crimes'
]

# Dynamically build the feature list from your columns
features = [col for col in balanced_train.columns if col not in exclude_cols]

print(f"Final Feature Count: {len(features)}")
print(f"Features included: {features}")

Final Feature Count: 26
Features included: ['gn_population', 'burglary', 'drugs', 'robbery', 'stabbing', 'theft', 'vehicle theft', 'Night', 'Morning', 'Afternoon', 'Evening', 'month', 'season_First_Inter', 'season_NE_Monsoon', 'season_SW_Monsoon', 'season_Second_Inter', 'has_holiday_in_week', 'weeks_from_start', 'crime_trend_4w', 'lag_count_1w', 'lag_count_2w', 'spatial_lag_1w', 'land_use_type_Commercial', 'land_use_type_General Urban', 'land_use_type_Recreational', 'land_use_type_Residential']


In [142]:
# 1. Identify your Time Cutoffs
latest_week = final_grid['week_start'].max()
test_week = latest_week - pd.Timedelta(weeks=1)

# 2. Slice the Data
inference_df = final_grid[final_grid['week_start'] == latest_week].copy()
test_df = final_grid[final_grid['week_start'] == test_week].copy()
train_df = final_grid[final_grid['week_start'] < test_week].copy()

# 3. Apply Balanced Sampling ONLY to the Training set
pos = train_df[train_df['target_next_week'] == 1]
neg = train_df[train_df['target_next_week'] == 0]
# Use min() to ensure you don't oversample if pos is very small
balanced_train = pd.concat([pos, neg.sample(n=len(pos) * 3, random_state=42)])

# 4. Prepare X and y
X_train = balanced_train[features]
y_train = balanced_train['target_next_week']

X_test = test_df[features]
y_test = test_df['target_next_week']

X_inference = inference_df[features]

### Feature Scaling

In [143]:
from sklearn.preprocessing import StandardScaler

# 1. Define only continuous numerical features
cols_to_scale = [
    'gn_population',
    'weeks_from_start',
    'lag_count_1w',
    'lag_count_2w',
    'spatial_lag_1w'
]

scaler = StandardScaler()

# 2. Fit ONLY on training data (to prevent data leakage)
scaler.fit(X_train[cols_to_scale])

# 3. Apply the transformation to all feature sets
X_train.loc[:, cols_to_scale] = scaler.transform(X_train[cols_to_scale])
X_test.loc[:, cols_to_scale] = scaler.transform(X_test[cols_to_scale])
X_inference.loc[:, cols_to_scale] = scaler.transform(X_inference[cols_to_scale])


/tmp/ipython-input-1633786257.py:18: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[-0.95782042 -0.95782042 -0.95782042 ...  1.57439211  0.06601001
 -0.57635621]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  X_train.loc[:, cols_to_scale] = scaler.transform(X_train[cols_to_scale])
/tmp/ipython-input-1633786257.py:18: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[-0.95032253 -0.27737128 -0.07421619 ...  1.10662278 -0.17579374
  1.43674981]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  X_train.loc[:, cols_to_scale] = scaler.transform(X_train[cols_to_scale])
/tmp/ipython-input-1633786257.py:19: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[-0.95782042  1.54079725 -0.29472833  0.72598645  0

In [144]:
X_train.info()

<class 'pandas.core.frame.DataFrame'>
Index: 13188 entries, 6833 to 26776
Data columns (total 26 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   gn_population                13188 non-null  float64
 1   burglary                     13188 non-null  float64
 2   drugs                        13188 non-null  float64
 3   robbery                      13188 non-null  float64
 4   stabbing                     13188 non-null  float64
 5   theft                        13188 non-null  float64
 6   vehicle theft                13188 non-null  float64
 7   Night                        13188 non-null  int64  
 8   Morning                      13188 non-null  int64  
 9   Afternoon                    13188 non-null  int64  
 10  Evening                      13188 non-null  int64  
 11  month                        13188 non-null  int32  
 12  season_First_Inter           13188 non-null  bool   
 13  season_NE_Monsoon 

Save the preprocessed data

In [145]:
# Save to Parquet for high-speed loading and data type preservation

X_train.to_parquet('/content/drive/MyDrive/DSGP/X_train.parquet')
y_train.to_frame().to_parquet('/content/drive/MyDrive/DSGP/y_train.parquet')

X_test.to_parquet('/content/drive/MyDrive/DSGP/X_test.parquet')
y_test.to_frame().to_parquet('/content/drive/MyDrive/DSGP/y_test.parquet')

# We save the full inference_df (not just X_inference)
# so we still have the 'gn_division' names for the dashboard map
inference_df.to_parquet('/content/drive/MyDrive/DSGP/inference_full.parquet')

print("Preprocessing complete. Files saved locally.")

Preprocessing complete. Files saved locally.
